# Notebook 01 — EDA y limpieza de la serie OUR (biológico-1)

**Proyecto RESPIR-AI** — predicción de la tasa de consumo de oxígeno (OUR, *Oxygen Uptake Rate*) en el reactor biológico de una EDAR.

Este cuaderno realiza el análisis exploratorio y la limpieza del dataset horario `biologico-1.parquet`:

1. **Perfilado**: rango temporal, estadísticos del OUR, disponibilidad por variable.
2. **Atípicos**: detección con z-score robusto (mediana/MAD en ventana móvil) más límites físicos. Los atípicos se **marcan y enmascaran**, nunca se borran filas.
3. **Huecos**: interpolación temporal solo de huecos ≤ 6 h; los huecos mayores dividen la serie en **segmentos contiguos** (`segment_id`), y los segmentos < 14 días (336 h) se excluyen del uso experimental.
4. **Dataset limpio**: `data/biologico-1_clean.parquet` con el OUR limpio, covariables, `segment_id` y los indicadores `our_outlier` / `our_imputed`.
5. **Estacionalidad**: perfiles por hora del día y día de la semana (hora local Europe/Madrid) y descomposición STL, calculados **sobre la serie ya limpia y restringida a segmentos válidos** — por eso esta sección va después de la limpieza.
6. **Figuras** para la memoria.

**Convención horaria**: el índice está en UTC; los perfiles de estacionalidad se calculan en hora local (Europe/Madrid) porque el ciclo de carga de la planta sigue la actividad humana local.

In [ ]:
# Raíz del repositorio como cwd (permite ejecutar desde notebooks/ o desde la raíz)
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/"common_eval.py").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import json
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

PARQUET = "data/biologico-1.parquet"
META = "data/biologico-1.meta.json"

# Nota: en este entorno pd.read_parquet falla por incompatibilidad pandas/pyarrow;
# se lee con pyarrow directamente.
df = pq.read_table(PARQUET).to_pandas()
meta = json.load(open(META))
print(df.shape)
df.head(3)

## 1. Perfilado general

El dataset cubre **21.156 horas** desde el 2023-08-31 22:00 UTC hasta el 2026-01-29 09:00 UTC, con **índice horario completo** (sin huecos de índice: los huecos son valores NaN, no filas ausentes). Contiene 34 columnas:

- **Target**: `our` (tasa de consumo de oxígeno).
- **Variables de proceso**: sólidos, SOUR, AUR, NUR, amonio/nitrato de salida, tiempos de aireación, etc. (~5.000–5.500 NaN cada una).
- **Soplantes** (`*_sopl*`): ~14.695 NaN — solo existen en el tramo final de la serie (instrumentación añadida después).
- **Meteorología** (6 variables: temperatura, humedad, precipitación, presión, viento, radiación): **completas**, sin NaN. Son las únicas covariables *futuras conocidas* utilizables en predicción.
- `latitude`/`longitude`: constantes (ubicación de la planta), sin valor predictivo.

In [ ]:
# Verificación de integridad del índice y disponibilidad por variable
idx_full = pd.date_range(df.index[0], df.index[-1], freq='h', tz='UTC')
assert len(idx_full) == len(df) and (df.index == idx_full).all(), "índice horario incompleto"
print(f"Rango: {df.index[0]} -> {df.index[-1]}  ({len(df)} horas)")

na = df.isna().sum().sort_values(ascending=False)
print(na.to_string())

In [ ]:
# Estadísticos del target
print(df['our'].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).round(3).to_string())
print("\nOUR <= 0:", int((df['our']<=0).sum()))

**Observaciones sobre el OUR crudo**: 15.530 valores observados (26,6 % NaN). Mediana 24,4 y p99 = 46,7, pero **máximo = 108,0**: más del doble del p99, lo que apunta a *spikes* instrumentales que la detección de atípicos debe tratar. No hay valores ≤ 0 (que serían físicamente inválidos).

Los NaN del OUR no están dispersos: se concentran en **101 rachas**, con cortes muy largos (1.082 h ≈ 45 días, 647 h, 639 h, 448 h, dos de 311 h…). Estas paradas del analizador hacen inviable imputar el target en esos tramos: la estrategia correcta es **segmentar**.

In [ ]:
# Rachas de NaN en OUR
isna = df['our'].isna()
grp = (isna != isna.shift()).cumsum()
run_info = pd.DataFrame({
    'isna': isna.groupby(grp).first(),
    'len': isna.groupby(grp).size(),
    'start': df.index.to_series().groupby(grp).first(),
    'end': df.index.to_series().groupby(grp).last(),
})
na_runs = run_info[run_info['isna']].sort_values('len', ascending=False)
print("Nº de rachas de NaN en OUR:", len(na_runs))
na_runs.head(10)

## 2. Detección de atípicos

Criterios (se **marcan** con `our_outlier=True` y se enmascaran a NaN; no se eliminan filas):

1. **Límites físicos**: OUR ≤ 0 es inválido (resultado: 0 casos).
2. **Z-score robusto en ventana móvil**: sobre una ventana centrada de 49 h se calculan la mediana y la MAD (desviación absoluta mediana); el z robusto es `0.6745·(x − mediana)/MAD` y se marca `|z| > 5`. Al usar mediana/MAD el estimador no se contamina por los propios spikes, y la ventana móvil lo hace adaptativo al nivel local de la serie.

**Resultado: 74 atípicos marcados** (0,48 % de los valores observados), entre ellos el máximo global de 108,0 (2026-01-08) y spikes aislados de 72, 66 y 62 frente a un p99 de 46,7. También se capturan desplomes puntuales a ~0,1 en mitad de tramos normales (caídas de sensor de una hora). El umbral |z|>5 es deliberadamente conservador: preferimos dejar pasar valores altos plausibles (puntas de carga reales de 45–50) antes que mutilar la variabilidad natural que los modelos deben aprender.

In [ ]:
our_raw = df['our'].copy()
phys_invalid = (our_raw <= 0)

win = 49
med = our_raw.rolling(win, center=True, min_periods=13).median()
mad = (our_raw - med).abs().rolling(win, center=True, min_periods=13).median()
mad_safe = mad.replace(0, np.nan).fillna(mad[mad>0].median())
z_rob = 0.6745 * (our_raw - med) / mad_safe
outlier_z = z_rob.abs() > 5.0

our_outlier = (phys_invalid | outlier_z) & our_raw.notna()
print("Atípicos marcados:", int(our_outlier.sum()),
      f"({our_outlier.sum()/our_raw.notna().sum()*100:.2f}% de los observados)")
df.loc[our_outlier, 'our'].sort_values(ascending=False).head(8)

## 3. Huecos y segmentación

Política de imputación, alineada con la práctica habitual en series de proceso:

- **Huecos ≤ 6 h**: interpolación temporal (lineal en el tiempo). Son microcortes de comunicación; la dinámica del OUR a esa escala es suave y el error introducido es pequeño. Se marcan con `our_imputed=True` (177 horas imputadas, 1,1 % de las horas útiles).
- **Huecos > 6 h**: NO se imputan. Inventar días o semanas de target contaminaría el entrenamiento y la evaluación. En su lugar, cada tramo contiguo de OUR válido recibe un `segment_id`; los huecos actúan como fronteras.
- **Segmentos < 14 días (336 h)**: se excluyen (`segment_id = -1`). Un tramo más corto no admite ventana de contexto + horizonte razonables para los modelos del estudio.

**Resultado: 17 segmentos válidos** con **13720 horas útiles** (572 días); se excluyen 38 tramos cortos que suman 1913 h.

In [ ]:
# Enmascarar atípicos e imputar huecos cortos
our_clean = our_raw.mask(our_outlier)
isna2 = our_clean.isna()
grp2 = (isna2 != isna2.shift()).cumsum()
run_len = isna2.groupby(grp2).transform('size')
short_gap = isna2 & (run_len <= 6)

our_interp = our_clean.interpolate(method='time', limit_area='inside')
our_final = our_clean.where(~short_gap, our_interp)
our_imputed = short_gap & our_final.notna()
print("Horas imputadas (huecos <=6h):", int(our_imputed.sum()))

# Segmentación
valid = our_final.notna()
seg_grp = (valid != valid.shift()).cumsum()
seg_id = pd.Series(-1, index=df.index, dtype=int)
seg_table, sid = [], 0
MIN_SEG = 336  # 14 días
for g, chunk in df.index.to_series().groupby(seg_grp):
    if not valid.loc[chunk.iloc[0]]:
        continue
    n = len(chunk)
    ok = n >= MIN_SEG
    seg_table.append({'segment_id': sid if ok else -1, 'inicio': chunk.iloc[0],
                      'fin': chunk.iloc[-1], 'horas': n, 'dias': round(n/24,1), 'valido': ok})
    if ok:
        seg_id.loc[chunk.index] = sid
        sid += 1
segs = pd.DataFrame(seg_table)
print("Segmentos válidos:", int(segs['valido'].sum()),
      "| horas útiles:", int(segs.loc[segs['valido'],'horas'].sum()))
segs[segs['valido']]

### Tabla de segmentos válidos

|   segment_id | inicio           | fin              |   horas |   dias |
|-------------:|:-----------------|:-----------------|--------:|-------:|
|            0 | 2023-10-06 10:00 | 2023-10-26 20:00 |     491 |   20.5 |
|            1 | 2023-11-08 12:00 | 2023-11-25 16:00 |     413 |   17.2 |
|            2 | 2024-03-05 11:00 | 2024-04-30 13:00 |    1347 |   56.1 |
|            3 | 2024-05-17 08:00 | 2024-06-12 17:00 |     634 |   26.4 |
|            4 | 2024-06-13 07:00 | 2024-09-11 17:00 |    2171 |   90.5 |
|            5 | 2024-09-13 08:00 | 2024-10-11 11:00 |     676 |   28.2 |
|            6 | 2024-10-16 12:00 | 2024-11-04 06:00 |     451 |   18.8 |
|            7 | 2024-11-05 08:00 | 2024-12-16 11:00 |     988 |   41.2 |
|            8 | 2025-01-17 10:00 | 2025-02-08 12:00 |     531 |   22.1 |
|            9 | 2025-02-09 15:00 | 2025-03-10 13:00 |     695 |   29   |
|           10 | 2025-03-13 13:00 | 2025-04-28 09:00 |    1101 |   45.9 |
|           11 | 2025-05-19 06:00 | 2025-06-04 10:00 |     389 |   16.2 |
|           12 | 2025-06-11 09:00 | 2025-08-04 12:00 |    1300 |   54.2 |
|           13 | 2025-08-07 06:00 | 2025-09-25 08:00 |    1179 |   49.1 |
|           14 | 2025-10-09 05:00 | 2025-11-04 10:00 |     630 |   26.2 |
|           15 | 2025-12-10 09:00 | 2025-12-25 10:00 |     362 |   15.1 |
|           16 | 2026-01-14 07:00 | 2026-01-29 08:00 |     362 |   15.1 |

### Tabla de exclusiones (tramos < 336 h, `segment_id = -1`)

| inicio           | fin              |   horas |   dias |
|:-----------------|:-----------------|--------:|-------:|
| 2023-08-31 22:00 | 2023-09-06 18:00 |     141 |    5.9 |
| 2023-10-03 10:00 | 2023-10-05 04:00 |      43 |    1.8 |
| 2023-10-27 13:00 | 2023-11-06 12:00 |     240 |   10   |
| 2023-11-26 09:00 | 2023-11-27 07:00 |      23 |    1   |
| 2023-11-29 16:00 | 2023-12-05 11:00 |     140 |    5.8 |
| 2024-01-19 14:00 | 2024-01-20 20:00 |      31 |    1.3 |
| 2024-01-21 05:00 | 2024-01-21 07:00 |       3 |    0.1 |
| 2024-01-21 21:00 | 2024-01-21 23:00 |       3 |    0.1 |
| 2024-01-23 16:00 | 2024-01-24 17:00 |      26 |    1.1 |
| 2024-01-25 22:00 | 2024-01-26 02:00 |       5 |    0.2 |
| 2024-01-26 16:00 | 2024-01-26 18:00 |       3 |    0.1 |
| 2024-01-27 02:00 | 2024-01-28 05:00 |      28 |    1.2 |
| 2024-01-28 13:00 | 2024-02-01 09:00 |      93 |    3.9 |
| 2024-02-05 15:00 | 2024-02-05 21:00 |       7 |    0.3 |
| 2024-02-06 11:00 | 2024-02-06 16:00 |       6 |    0.2 |
| 2024-02-07 17:00 | 2024-02-09 09:00 |      41 |    1.7 |
| 2024-02-10 14:00 | 2024-02-14 00:00 |      83 |    3.5 |
| 2024-02-14 13:00 | 2024-02-23 08:00 |     212 |    8.8 |
| 2024-02-23 17:00 | 2024-02-23 22:00 |       6 |    0.2 |
| 2024-02-26 09:00 | 2024-02-26 14:00 |       6 |    0.2 |
| 2024-02-27 09:00 | 2024-02-28 21:00 |      37 |    1.5 |
| 2024-02-29 08:00 | 2024-03-03 04:00 |      69 |    2.9 |
| 2024-05-02 16:00 | 2024-05-03 06:00 |      15 |    0.6 |
| 2024-05-07 13:00 | 2024-05-13 01:00 |     133 |    5.5 |
| 2024-05-13 11:00 | 2024-05-16 18:00 |      80 |    3.3 |
| 2024-09-12 11:00 | 2024-09-12 23:00 |      13 |    0.5 |
| 2024-12-17 10:00 | 2024-12-19 12:00 |      51 |    2.1 |
| 2025-01-15 12:00 | 2025-01-15 16:00 |       5 |    0.2 |
| 2025-01-16 11:00 | 2025-01-16 16:00 |       6 |    0.2 |
| 2025-04-30 09:00 | 2025-04-30 13:00 |       5 |    0.2 |
| 2025-06-04 20:00 | 2025-06-05 13:00 |      18 |    0.8 |
| 2025-08-05 10:00 | 2025-08-06 09:00 |      24 |    1   |
| 2025-10-02 09:00 | 2025-10-03 12:00 |      28 |    1.2 |
| 2025-11-17 10:00 | 2025-11-18 14:00 |      29 |    1.2 |
| 2025-11-20 13:00 | 2025-11-26 09:00 |     141 |    5.9 |
| 2025-12-01 15:00 | 2025-12-04 10:00 |      68 |    2.8 |
| 2025-12-05 10:00 | 2025-12-05 13:00 |       4 |    0.2 |
| 2026-01-07 10:00 | 2026-01-09 08:00 |      47 |    2   |

Los tramos excluidos suman 1913 h (80 días). Nótese que el invierno 2023-24 (dic–feb) queda casi enteramente excluido: el corte de 1.082 h y la posterior intermitencia de enero–febrero de 2024 no dejan ningún tramo de 14 días contiguos.

## 4. Dataset limpio de salida

`data/biologico-1_clean.parquet` conserva las 21.156 filas y todas las columnas originales, más:

| Columna | Tipo | Significado |
|---|---|---|
| `our` | float | OUR limpio: atípicos enmascarados (NaN), huecos ≤6 h imputados |
| `our_outlier` | bool | el valor original fue marcado como atípico y enmascarado |
| `our_imputed` | bool | el valor fue rellenado por interpolación temporal |
| `segment_id` | int | segmento contiguo válido (0…16); −1 = hora no utilizable |

El experimento debe usar exclusivamente las filas con `segment_id >= 0` y nunca cruzar fronteras de segmento al construir ventanas.

In [ ]:
df_clean = df.copy()
df_clean['our'] = our_final
df_clean['our_outlier'] = our_outlier.astype(bool)
df_clean['our_imputed'] = our_imputed.astype(bool)
df_clean['segment_id'] = seg_id.values
df_clean.to_parquet('data/biologico-1_clean.parquet')
print("Guardado data/biologico-1_clean.parquet:", df_clean.shape)

## 5. Estacionalidad diaria y semanal (sobre la serie limpia)

Con el dataset ya depurado podemos caracterizar la estacionalidad sin que los spikes ni los tramos degradados la distorsionen. Los perfiles de mediana y rango intercuartílico por **hora del día** y **día de la semana** (hora local Europe/Madrid) se calculan sobre las **horas útiles** (`segment_id >= 0`) del OUR limpio.

In [ ]:
our_util = df_clean.loc[df_clean['segment_id'] >= 0, 'our']
loc = our_util.index.tz_convert('Europe/Madrid')
prof_hora = our_util.groupby(loc.hour).agg(['mean','median',
    lambda s: s.quantile(.25), lambda s: s.quantile(.75)])
prof_hora.columns = ['media','mediana','q25','q75']
prof_dia = our_util.groupby(loc.dayofweek).agg(['mean','median',
    lambda s: s.quantile(.25), lambda s: s.quantile(.75)])
prof_dia.columns = ['media','mediana','q25','q75']
prof_hora.round(2)

**Resultado**: existe un **ciclo diario suave** — máximo matinal hacia las 6–8 h locales (mediana ≈ 25,4) y mínimo nocturno hacia las 22–23 h (mediana ≈ 23,3), con amplitud mediana de ~2 mg O₂/L·h — coherente con el ciclo de carga orgánica del agua de entrada. El **patrón semanal es prácticamente plano** (medianas entre 23,9 y 24,4 según el día): la planta no muestra el contraste laborable/festivo típico de series urbanas de consumo.

### Descomposición STL

STL (periodo 24 h, robusta) sobre el **segmento válido más largo** (segmento 4: 2.171 h, jun–sep 2024, sin NaN por construcción tras la limpieza).

In [ ]:
seg4 = df_clean.loc[df_clean['segment_id'] == 4, 'our']
print("Segmento 4:", seg4.index[0], "->", seg4.index[-1], "| NaN:", int(seg4.isna().sum()))
stl_res = STL(seg4, period=24, robust=True).fit()
fuerza = max(0, 1 - stl_res.resid.var()/(stl_res.resid + stl_res.seasonal).var())
print(f"Fuerza estacional diaria: {fuerza:.3f}")
print(f"Amplitud estacional (p95-p5): {stl_res.seasonal.quantile(.95)-stl_res.seasonal.quantile(.05):.2f}")
print(f"std residuo: {stl_res.resid.std():.2f} | std serie: {seg4.std():.2f}")
fig = stl_res.plot(); fig.set_size_inches(10, 7); plt.tight_layout()

La STL cuantifica la impresión de los perfiles: **fuerza estacional diaria ≈ 0,18** y amplitud del componente estacional (p95–p5) ≈ 9,7 frente a una desviación típica de la serie de 11,8. Es decir, la estacionalidad diaria existe pero explica una fracción menor de la varianza — el OUR está dominado por dinámica no periódica (carga, proceso, control de aireación), lo que anticipa que los modelos no podrán apoyarse solo en el ciclo diario.

## 6. Figuras para la memoria

Se generan tres figuras (en `figures/`):

1. `panorama_serie_segmentos.png` — serie completa con los 17 segmentos válidos sombreados y los 74 atípicos enmascarados.
2. `perfil_diario_semanal.png` — mediana y rango intercuartílico por hora del día (local) y día de la semana.
3. `distribucion_our.png` — histograma del OUR limpio en horas útiles, con mediana y p99.

In [ ]:
import os
os.makedirs('figures', exist_ok=True)
fig, ax = plt.subplots(figsize=(12, 3.6))
ax.plot(df_clean.index, df_clean['our'], lw=0.4, color='#33546e')
for _, s in segs[segs['valido']].iterrows():
    ax.axvspan(s['inicio'], s['fin'], color='#8fbf9f', alpha=0.28, lw=0)
out_pts = df.loc[our_outlier, 'our']
ax.plot(out_pts.index, out_pts.values, 'o', ms=2.5, color='#c0392b')
ax.set_ylabel('OUR (mg O₂/L·h)')
ax.set_title('La serie OUR queda dividida en 17 segmentos útiles (sombreado) por los huecos largos del sensor', loc='left')
fig.savefig('figures/panorama_serie_segmentos.png', dpi=300, bbox_inches='tight')

In [ ]:
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True,
                                gridspec_kw={'wspace':0.06, 'width_ratios':[2,1]})
ax1.fill_between(prof_hora.index, prof_hora['q25'], prof_hora['q75'], color='#33546e', alpha=0.18, lw=0)
ax1.plot(prof_hora.index, prof_hora['mediana'], color='#33546e', lw=1.6, marker='o', ms=3)
ax1.set_xlabel('hora del día (Europe/Madrid)'); ax1.set_ylabel('OUR (mg O₂/L·h)')
ax1.set_title('Ciclo diario suave: máximo matinal (6–8 h) y mínimo nocturno', loc='left')
ax2.fill_between(prof_dia.index, prof_dia['q25'], prof_dia['q75'], color='#8a6d3b', alpha=0.18, lw=0)
ax2.plot(prof_dia.index, prof_dia['mediana'], color='#8a6d3b', lw=1.6, marker='o', ms=3)
ax2.set_xticks(range(7)); ax2.set_xticklabels(['L','M','X','J','V','S','D'])
ax2.set_xlabel('día de la semana'); ax2.set_title('Sin patrón semanal marcado', loc='left')
fig2.savefig('figures/perfil_diario_semanal.png', dpi=300, bbox_inches='tight')

In [ ]:
fig3, ax = plt.subplots(figsize=(6, 3.2))
vals = df_clean.loc[df_clean['segment_id']>=0, 'our'].dropna()
ax.hist(vals, bins=60, color='#33546e', alpha=0.85, edgecolor='white', lw=0.3)
for q, lab in [(vals.median(), f'mediana = {vals.median():.1f}'),
               (vals.quantile(.99), f'p99 = {vals.quantile(.99):.1f}')]:
    ax.axvline(q, color='#8a6d3b', lw=1, ls='--')
    ax.text(q+0.6, ax.get_ylim()[1]*0.92, lab, fontsize=8, color='#8a6d3b')
ax.set_xlabel('OUR (mg O₂/L·h)'); ax.set_ylabel('nº de horas')
ax.set_title('Distribución unimodal y ligeramente asimétrica del OUR limpio', loc='left')
fig3.savefig('figures/distribucion_our.png', dpi=300, bbox_inches='tight')

## Conclusiones del EDA

- La serie es **horaria, completa en índice**, pero con un 26,6 % del target ausente en rachas largas: la segmentación es obligatoria.
- Quedan **17 segmentos válidos / 13.720 h útiles** (~572 días), suficientes para una partición 70/10/20 y evaluación rolling-origin.
- **74 atípicos** enmascarados (0,48 %) y **177 h** imputadas en huecos ≤ 6 h; ambos trazables por columna de flag.
- Estacionalidad diaria **débil** (fuerza ≈ 0,18) y semanal casi nula: los modelos deberán explotar la autocorrelación de corto plazo y las covariables, no un ciclo periódico fuerte.
- Las 6 variables meteorológicas están completas y pueden usarse como covariables futuras conocidas; las columnas de soplantes solo existen en el tramo final y no son utilizables como covariable global.